# Interactive Bounding-Box Annotation in a Jupyter Notebook

Using the [`ipycanvas`](https://github.com/martinRenou/ipycanvas) library, which embeds an HTML5 `<canvas>` in the cell output we will create a interactive metrics calculator for Jupyter notebooks. The canvas receives real mouse events, so we can:

- display an image,
- draw a fixed **ground-truth** box in one colour,
- let you **click-and-drag** to create a **prediction** box in a different colour,
- **drag the prediction box** around afterwards to refine it,
- and live-compute the **IoU** between the two boxes.

## 1. Install / imports

If you don't have `ipycanvas` yet:

```
pip install ipycanvas numpy Pillow
```

In [ ]:
#!pip install ipycanvas

In [4]:
import numpy as np
from IPython.display import display, Javascript
from ipycanvas import Canvas, hold_canvas
from ipywidgets import VBox, HTML, Button

## 2. Image & ground-truth box

The notebook builds a small synthetic image so it works offline. To use your own image, replace `img` with an `H×W×3` `uint8` NumPy array (e.g. `np.asarray(Image.open('your.jpg'))`) and update `gt_box`.

In [5]:
H, W = 400, 600
img = np.full((H, W, 3), 235, dtype=np.uint8)

body_color, roof_color = (200, 50, 50), (170, 30, 30)
wheel_color, hub_color = (30, 30, 30), (140, 140, 140)
img[200:285, 150:455] = body_color
img[160:200, 220:385] = roof_color
img[270:300, 180:225] = wheel_color
img[270:300, 385:430] = wheel_color
img[280:295, 192:215] = hub_color
img[280:295, 397:420] = hub_color

gt_box = [140, 150, 325, 160]
img_rgba = np.dstack([img, np.full((H, W, 1), 255, dtype=np.uint8)])

## 3. The interactive widget

- **Click & drag** on the image to draw the prediction box (green, dashed).
- Once drawn, **click & drag inside it** to move it around.
- **Mouse wheel** resizes the box (±5 px on w and h, from its centre).
- The **IoU**, **1−IoU loss** and **Huber loss** with the red ground-truth box update live.
- Use the *Reset prediction* button to start over.

In [4]:
GT_COLOR   = '#FF1744'
PRED_COLOR = '#00E676'

state = {
    'pred_box': None,
    'mode'    : 'idle',
    'start'   : None,
    'offset'  : None,
}

def draw_scene():
    canvas.put_image_data(img_rgba, 0, 0)

    x, y, w, h = gt_box
    canvas.stroke_style = GT_COLOR
    canvas.line_width   = 3
    canvas.stroke_rect(x, y, w, h)
    canvas.fill_style = GT_COLOR
    canvas.font = 'bold 16px sans-serif'
    canvas.fill_text('Ground Truth', x, y - 6)

    if state['pred_box'] is not None:
        x, y, w, h = state['pred_box']
        canvas.stroke_style = PRED_COLOR
        canvas.line_width   = 3
        canvas.set_line_dash([8, 4])
        canvas.stroke_rect(x, y, w, h)
        canvas.set_line_dash([])
        canvas.fill_style = PRED_COLOR
        canvas.fill_text('Prediction (drag me)', x, y - 6)

# implement this function a and p are tuples (x,y,l,w)
def iou(a, b):
    return 1

# implement this function pred and gt are tuples (x,y,l,w)
def huber(pred, gt, delta=1.0):
    return 1

def box_deltas(pred, gt):
    return [p - g for p, g in zip(pred, gt)]


def hit(box, mx, my):
    if box is None: return False
    x, y, w, h = box
    return x <= mx <= x+w and y <= my <= y+h

def clamp(box):
    x, y, w, h = box
    x = max(0, min(W - w, x))
    y = max(0, min(H - h, y))
    return [x, y, w, h]

def update_iou():
    if state['pred_box'] is None:
        info.value = ('<b>IoU:</b> — &nbsp;·&nbsp; <b>1−IoU loss:</b> — &nbsp;·&nbsp; '
                      '<b>Huber loss:</b> —<br>'
                      '<i>click &amp; drag on the image to draw a prediction box</i>')
    else:
        v = iou(state['pred_box'], gt_box)
        dx, dy, dw, dh = box_deltas(state['pred_box'], gt_box)
        h = huber(state['pred_box'], gt_box)
        info.value = (f'<b>IoU:</b> {v:.3f} &nbsp;·&nbsp; '
                      f'<b>1−IoU loss:</b> {1-v:.3f} &nbsp;·&nbsp; '
                      f'<b>Huber loss:</b> {h:.3f}<br>'
                      f'<b>Prediction [x,y,w,h]:</b> ({state["pred_box"][0]:.0f}, {state["pred_box"][1]:.0f}, '
                      f'{state["pred_box"][2]:.0f}, {state["pred_box"][3]:.0f}) &nbsp; '
                      f'<b>Δ(x,y,w,h):</b> ({dx:+.0f}, {dy:+.0f}, {dw:+.0f}, {dh:+.0f})')

def on_down(x, y):
    if hit(state['pred_box'], x, y):
        state['mode']   = 'moving'
        state['offset'] = (x - state['pred_box'][0], y - state['pred_box'][1])
    else:
        state['mode']    = 'drawing'
        state['start']   = (x, y)
        state['pred_box'] = [x, y, 0, 0]

def on_move(x, y):
    if state['mode'] == 'drawing':
        sx, sy = state['start']
        state['pred_box'] = [min(sx, x), min(sy, y), abs(x - sx), abs(y - sy)]
    elif state['mode'] == 'moving':
        ox, oy = state['offset']
        _, _, pw, ph = state['pred_box']
        state['pred_box'] = clamp([x - ox, y - oy, pw, ph])
    else:
        return
    with hold_canvas(canvas):
        draw_scene()
    update_iou()

def on_up(x, y):
    if state['mode'] == 'drawing':
        if state['pred_box'][2] < 5 or state['pred_box'][3] < 5:
            state['pred_box'] = None
    state['mode']   = 'idle'
    state['start']  = None
    state['offset'] = None
    with hold_canvas(canvas):
        draw_scene()
    update_iou()

def reset(_):
    state['pred_box'] = None
    state['mode']     = 'idle'
    with hold_canvas(canvas):
        draw_scene()
    update_iou()

def on_wheel(*args, **kwargs):
    if state['pred_box'] is None:
        return
    delta_y = kwargs.get('delta_y', args[-1] if args else 0)
    if not delta_y:
        return
    px, py, pw, ph = state['pred_box']
    step = -int(np.sign(delta_y)) * 5
    new_w = max(5, min(W, pw + step))
    new_h = max(5, min(H, ph + step))
    cx = px + pw / 2
    cy = py + ph / 2
    state['pred_box'] = [
        max(0, min(W - new_w, cx - new_w / 2)),
        max(0, min(H - new_h, cy - new_h / 2)),
        new_w, new_h,
    ]
    with hold_canvas(canvas):
        draw_scene()
    update_iou()

canvas = Canvas(width=W, height=H)
canvas.on_mouse_down(on_down)
canvas.on_mouse_move(on_move)
canvas.on_mouse_up(on_up)
canvas.on_mouse_wheel(on_wheel)

info      = HTML()
reset_btn = Button(description='Reset prediction', button_style='warning', icon='trash')
reset_btn.on_click(reset)

with hold_canvas(canvas):
    draw_scene()
update_iou()

display(Javascript("""
(function() {
    document.querySelectorAll('canvas').forEach(c => {
        if (!c._wheelFixed) {
            c.addEventListener('wheel', e => e.preventDefault(), {passive: false});
            c._wheelFixed = true;
        }
    });
})();
"""))

canvas.layout.width  = f'{W}px'
canvas.layout.height = f'{H}px'
ui = VBox([info, canvas, reset_btn])
ui.layout.min_width  = f'{W + 20}px'
ui

<IPython.core.display.Javascript object>

## 4. Exercise

Implement the two following functions yourself and replace the provided implementations with yours:

- **`iou(a, b)`** — Intersection-over-Union of two boxes `[x, y, w, h]`.
- **`huber(pred, gt, delta=1.0)`** — Huber / Smooth-L1 loss over the four box coordinates.

Hints:
- Boxes use the `[x, y, w, h]` convention (top-left corner + width/height).
- IoU: compute the area of the intersection rectangle clipped to the boxes, divide by the union.
- Huber: L(a) = 0.5·a²/δ when |a| ≤ δ, else δ·(|a| − 0.5·δ). Average over the 4 coordinates.

Test your implementation by drawing a few boxes and comparing the live values above with a hand calculation.